# 002: Training Pipeline Architecture — Iteration loops, early stopping, and checkpointing

## 1. THE ITERATION PROCESS
- **Epoch**: One complete forward and backward pass of the entire training dataset.
- **Batch/Step**: An update of parameters over one mini-batch of data.
- **Validation Loop**: Periodically evaluates the model on unseen validation data without computing gradients.

## 2. REGULARIZATION & SAVING
- **Early Stopping**: Tracks validation loss. If it fails to improve for a predefined number of epochs (the *patience* threshold), training halts early. This prevents overfitting when training starts to memorize noise.
- **Checkpointing**: Saves the best-performing model weights dynamically during training, ensuring that the model from the best validation state is kept.
---


In [ ]:
import numpy as np
import copy

class TrainingPipeline:
    """Manages training iterations, early stopping, and checkpoint saving from scratch."""
    def __init__(self, model, optimizer, patience=5):
        self.model = model
        self.optimizer = optimizer
        self.patience = patience
        self.best_loss = np.inf
        self.best_weights = None
        self.patience_counter = 0

    def fit(self, X_train, y_train, X_val, y_val, epochs=50, batch_size=32):
        train_losses = []
        val_losses = []
        
        for epoch in range(epochs):
            # 1. Shuffle data
            indices = np.arange(X_train.shape[1])
            np.random.shuffle(indices)
            X_shuf = X_train[:, indices]
            y_shuf = y_train[:, indices]
            
            epoch_loss = 0.0
            num_batches = 0
            
            # 2. Mini-batch Training Loop
            for i in range(0, X_train.shape[1], batch_size):
                X_batch = X_shuf[:, i:i+batch_size]
                y_batch = y_shuf[:, i:i+batch_size]
                
                # Forward, loss, backward, update
                y_pred = self.model.forward(X_batch)
                loss = self.model.compute_loss(y_batch, y_pred)
                epoch_loss += loss
                num_batches += 1
                
                grads = self.model.backward(X_batch, y_batch)
                self.optimizer.update(self.model.params, grads)
                
            train_loss = epoch_loss / num_batches
            train_losses.append(train_loss)
            
            # 3. Validation Step
            y_val_pred = self.model.forward(X_val)
            val_loss = self.model.compute_loss(y_val, y_val_pred)
            val_losses.append(val_loss)
            
            # 4. Checkpoint & Early Stopping evaluation
            if val_loss < self.best_loss:
                self.best_loss = val_loss
                self.best_weights = copy.deepcopy(self.model.params)
                self.patience_counter = 0
            else:
                self.patience_counter += 1
                
            if epoch % 5 == 0 or epoch == epochs - 1:
                print(f"Epoch {epoch:2d} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} | Patience: {self.patience_counter}/{self.patience}")
                
            if self.patience_counter >= self.patience:
                print(f"Early stopping triggered at epoch {epoch}. Restoring best weights with Val Loss: {self.best_loss:.6f}")
                self.model.params = self.best_weights
                break
                
        return train_losses, val_losses



In [ ]:
# Mock model and optimizer for testing early stopping
class MockLinearModel:
    def __init__(self):
        self.params = {"w": np.array([[1.5]])}
        
    def forward(self, X):
        return self.params["w"] @ X
        
    def compute_loss(self, y, y_pred):
        return np.mean((y - y_pred) ** 2)
        
    def backward(self, X, y):
        # Return static gradients that lead to poor validation loss to trigger early stopping
        return {"w": np.array([[0.1]])}

class MockOptimizer:
    def update(self, params, grads):
        params["w"] -= 0.1 * grads["w"]



In [ ]:
print("--- Training Loop with Early Stopping ---")
np.random.seed(42)
X_tr = np.random.randn(1, 100)
y_tr = X_tr * 2.0
# Val targets are set differently to trigger validation error and early stopping
X_va = np.random.randn(1, 30)
y_va = X_va * -1.0 

pipeline = TrainingPipeline(MockLinearModel(), MockOptimizer(), patience=3)
pipeline.fit(X_tr, y_tr, X_va, y_va, epochs=30, batch_size=10)
